# 🎬 LTX 2.5-MSR Ai Blueprint — Gradio Live UI
**Cell 1**: Cài đặt + tải models → **Cell 2**: Chạy Gradio UI (gradio.live)

In [ ]:
#@title 📦 Cell 1: Install ComfyUI + Custom Nodes + Download Models
import concurrent.futures
import os
import subprocess
import threading
import time
from pathlib import Path
from IPython.display import display, HTML

def log(msg, color="#00e676"):
    display(HTML(f"<p style='color:{color}; font-weight:bold;'>{msg}</p>"))

def sh(cmd):
    get_ipython().system(cmd)

# ============================================================
# [0/4] HuggingFace Token
# ============================================================
log("[0/4] Kiểm tra Hugging Face Token...")

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    if HF_TOKEN:
        log("✅ Lấy HF_TOKEN từ Colab Secrets thành công.")
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    from getpass import getpass
    log(
        "🔑 Chưa tìm thấy HF_TOKEN. Dán access token vào ô bên dưới.\n"
        "Lấy token tại: https://huggingface.co/settings/tokens\n"
        "Và bấm 'Agree' tại: https://huggingface.co/Lightricks/LTX-2.5",
        color="#ffb300",
    )
    HF_TOKEN = getpass("Dán HF token rồi nhấn Enter: ").strip()

if not HF_TOKEN:
    raise RuntimeError("❌ Không có HF_TOKEN → không tải được model (repo gated).")

os.environ["HF_TOKEN"] = HF_TOKEN
AUTH_HEADER = f"Authorization: Bearer {HF_TOKEN}"

# ============================================================
# [1/4] ComfyUI + Core deps
# ============================================================
log("[1/4] Installing ComfyUI...")

if not os.path.exists("/content/ComfyUI"):
    sh("git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI")
else:
    log("✅ ComfyUI OK")

sh("pip install -q -r /content/ComfyUI/requirements.txt")
sh("apt-get -y install -qq aria2 > /dev/null 2>&1")
sh("pip install -q gradio opencv-python")

# ============================================================
# [2/4] Custom Nodes
# ============================================================
log("[2/4] Installing Custom Nodes...")

CUSTOM_NODES = [
    "https://github.com/kijai/ComfyUI-KJNodes",
    "https://github.com/kijai/ComfyUI-PromptRelay",
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite",
    "https://github.com/Lightricks/ComfyUI-LTXVideo/",
    "https://github.com/liconstudio/ComfyUI-LTX2.5-MSR",
    "https://github.com/city96/ComfyUI-GGUF",
]
CN_DIR = "/content/ComfyUI/custom_nodes"
for repo_url in CUSTOM_NODES:
    name = repo_url.rstrip("/").split("/")[-1]
    p = os.path.join(CN_DIR, name)
    if os.path.exists(p):
        print(f"  ✅ {name}")
    else:
        print(f"  📥 {name}...")
        sh(f"git clone -q {repo_url} {p}")
    rf = os.path.join(p, "requirements.txt")
    if os.path.exists(rf):
        sh(f"pip install -q -r {rf}")

# ============================================================
# [3/4] Download Models
# ============================================================
log("[3/4] Downloading models (~40GB)...")

_FAILED = []
_lock = threading.Lock()

UNET_FILENAME             = "ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors"
TEXT_ENCODER_FILENAME     = "gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors"
VIDEO_VAE_FILENAME        = "ltx-2.5-video-vae-bf16.safetensors"
AUDIO_VAE_FILENAME        = "ltx-2.5-audio-vae-bf16.safetensors"
SPATIAL_UPSCALER_FILENAME = "ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors"
MSR_LORA_FILENAME         = "LTX-2.5-Licon-MSR-V1.safetensors"

def dl(url, dest, fname, connections=8, gated=False):
    Path(dest).mkdir(parents=True, exist_ok=True)
    fp = os.path.join(dest, fname)
    if os.path.exists(fp):
        with _lock:
            print(f"  ✅ {fname}")
        return True
    with _lock:
        print(f"  ⬇️  {fname}...")
    cmd = [
        "aria2c", "--console-log-level=warn", "-c",
        "-x", str(connections), "-s", str(connections), "-k", "1M",
        "-d", dest, "-o", fname,
    ]
    if gated:
        cmd += ["--header", AUTH_HEADER]
    cmd.append(url)
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = max(time.time() - t0, 0.01)
    ok = result.returncode == 0 and os.path.exists(fp)
    with _lock:
        if ok:
            sz = os.path.getsize(fp) / (1024*1024)
            print(f"  ✅ {fname} ({sz:.0f}MB, {elapsed:.0f}s, ~{sz/elapsed:.1f}MB/s)")
        else:
            _FAILED.append(fname)
            hint = ""
            if gated and ("401" in (result.stderr or "") or "403" in (result.stderr or "")):
                hint = " — kiểm tra HF_TOKEN + đã accept license repo"
            print(f"  ❌ {fname}{hint}")
    return ok

JOBS = [
    (f"https://huggingface.co/Lightricks/LTX-2.5/resolve/main/diffusion_models/{UNET_FILENAME}",
     "/content/ComfyUI/models/diffusion_models", UNET_FILENAME, True),
    (f"https://huggingface.co/Lightricks/LTX-2.5/resolve/main/text_encoders/{TEXT_ENCODER_FILENAME}",
     "/content/ComfyUI/models/text_encoders", TEXT_ENCODER_FILENAME, True),
    (f"https://huggingface.co/Lightricks/LTX-2.5/resolve/main/vae/{VIDEO_VAE_FILENAME}",
     "/content/ComfyUI/models/vae", VIDEO_VAE_FILENAME, True),
    (f"https://huggingface.co/Lightricks/LTX-2.5/resolve/main/vae/{AUDIO_VAE_FILENAME}",
     "/content/ComfyUI/models/vae", AUDIO_VAE_FILENAME, True),
    (f"https://huggingface.co/Lightricks/LTX-2.5/resolve/main/latent_upscale_models/{SPATIAL_UPSCALER_FILENAME}",
     "/content/ComfyUI/models/latent_upscale_models", SPATIAL_UPSCALER_FILENAME, True),
    (f"https://huggingface.co/LiconStudio/LTX-2.5-Multiple-Subject-Reference/resolve/main/{MSR_LORA_FILENAME}",
     "/content/ComfyUI/models/loras/ltx2.5", MSR_LORA_FILENAME, False),
]

with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(dl, url, dest, fname, 8, gated) for url, dest, fname, gated in JOBS]
    concurrent.futures.wait(futures)

os.makedirs("/content/ComfyUI/input", exist_ok=True)

# ============================================================
# [4/4] Summary
# ============================================================
if _FAILED:
    log(f"⚠️ {len(_FAILED)} file lỗi: {', '.join(_FAILED)}. Kiểm tra HF_TOKEN rồi chạy lại.", color="#ff5252")
else:
    log("✅ TẤT CẢ ĐÃ SẴN SÀNG! Chạy Cell 2 ➜ Gradio Live")

In [ ]:
#@title 🎬 Cell 2: Chạy Gradio UI — Auto Generate Video
get_ipython().system("pip install -q gradio opencv-python")

import glob
import json
import math
import os
import random
import re
import shutil
import socket
import subprocess
import time
import urllib.request

import cv2
import gradio as gr

# ==========================================================================
# CONFIG
# ==========================================================================
INPUT_DIR  = "/content/ComfyUI/input/"
OUTPUT_DIR = "/content/ComfyUI/output/"
LORA_DIR   = "/content/ComfyUI/models/loras/"
MSR_LORA_SUBDIR = "ltx2.5"

UNET_FILENAME             = "ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors"
TEXT_ENCODER_FILENAME     = "gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors"
VIDEO_VAE_FILENAME        = "ltx-2.5-video-vae-bf16.safetensors"
AUDIO_VAE_FILENAME        = "ltx-2.5-audio-vae-bf16.safetensors"
SPATIAL_UPSCALER_FILENAME = "ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors"
MSR_LORA_FILENAME         = "ltx2.5/LTX-2.5-Licon-MSR-V1.safetensors"

LATENT_GROUP_FRAMES = 8
SIGMAS_PASS1 = "1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0"
SIGMAS_PASS2 = "0.85, 0.7250, 0.4219, 0.0"

NEGATIVE_PROMPT_DEFAULT = (
    "blurry, oversaturated, pixelated, low resolution, grainy, distorted, noise, "
    "compression artifacts, glitches, watermark, text, logo, subtitles, "
    "static frame, frozen image, standing still, lack of motion, ignored prompt, "
    "deformed limbs, extra paws, duplicate limbs, distorted face, inconsistent appearance, "
    "mid-shot camera cut, character switching, sudden transition"
)

# ==========================================================================
# SERVER MANAGEMENT
# ==========================================================================
def is_server_running(port=8188):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

_SERVER_STATE = {"running_low_vram": None, "custom_nodes_mtime": None}

def _get_custom_nodes_mtime():
    cn_dir = "/content/ComfyUI/custom_nodes"
    try:
        mtimes = []
        for entry in os.scandir(cn_dir):
            mtimes.append(entry.stat().st_mtime)
            if entry.is_dir():
                try:
                    for sub in os.scandir(entry.path):
                        if sub.name.endswith(".py"):
                            mtimes.append(sub.stat().st_mtime)
                except OSError:
                    pass
        return max(mtimes) if mtimes else 0.0
    except OSError:
        return 0.0

def ensure_server(low_vram=True, boot_timeout=300):
    current_mtime = _get_custom_nodes_mtime()
    need_restart = (
        not is_server_running()
        or _SERVER_STATE["running_low_vram"] != low_vram
        or _SERVER_STATE["custom_nodes_mtime"] != current_mtime
    )
    if not need_restart:
        return
    os.system("fuser -k 8188/tcp")
    time.sleep(2)
    os.chdir("/content/ComfyUI")
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    cmd = ["python", "main.py"]
    if low_vram:
        cmd.insert(2, "--cache-none")
    subprocess.Popen(cmd)
    waited = 0
    while not is_server_running():
        time.sleep(2)
        waited += 2
        if waited > boot_timeout:
            raise RuntimeError(f"Server khong khoi dong duoc sau {boot_timeout}s.")
    _SERVER_STATE["running_low_vram"] = low_vram
    _SERVER_STATE["custom_nodes_mtime"] = current_mtime

# ==========================================================================
# HELPERS
# ==========================================================================
def snap_fps_safe(fps):
    try:
        fps = float(fps)
    except (TypeError, ValueError):
        fps = 24.0
    safe = int(round(fps / LATENT_GROUP_FRAMES) * LATENT_GROUP_FRAMES)
    return max(LATENT_GROUP_FRAMES, safe)

def half_dims(width, height):
    def snap_half_up(v):
        v = int(v)
        return max(32, int(math.ceil(v / 2.0 / 32.0)) * 32)
    return snap_half_up(width), snap_half_up(height)

def safe_dims(width, height):
    def snap_up(v):
        return max(256, int(math.ceil(int(v) / 32.0)) * 32)
    return snap_up(width), snap_up(height)

def get_seed(v_seed):
    try:
        v = int(v_seed)
    except Exception:
        v = -1
    return random.randint(1, 999_999_999) if v == -1 else v

def parse_aspect_ratio(ratio_str):
    if "480x832"  in ratio_str: return 480,  832
    if "832x480"  in ratio_str: return 832,  480
    if "1280x720" in ratio_str: return 1280, 720
    if "720x1280" in ratio_str: return 720,  1280
    if "720x720"  in ratio_str: return 720,  720
    return 512, 512

def list_msr_loras():
    msr_dir = os.path.join(LORA_DIR, MSR_LORA_SUBDIR)
    os.makedirs(msr_dir, exist_ok=True)
    files = sorted(
        f"{MSR_LORA_SUBDIR}/{f}"
        for f in os.listdir(msr_dir)
        if f.lower().endswith((".safetensors", ".pt", ".ckpt"))
    )
    return files if files else [MSR_LORA_FILENAME]

def find_latest_video(output_dir=OUTPUT_DIR):
    mp4_files = (
        glob.glob(f"{output_dir}*.mp4")
        + glob.glob(f"{output_dir}output/*.mp4")
        + glob.glob(f"{output_dir}video/*.mp4")
    )
    if not mp4_files:
        return None
    return max(mp4_files, key=os.path.getmtime)

def get_video_duration(video_path):
    cmd = ["ffprobe", "-v", "error", "-select_streams", "v:0",
           "-show_entries", "format=duration", "-of", "csv=p=0", video_path]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        return float(result.stdout.strip())
    except Exception:
        return None

def trim_ref_frames(video_path, target_duration_s, fps, output_dir=OUTPUT_DIR):
    actual = get_video_duration(video_path)
    if actual is None:
        return video_path
    margin = 1.0 / max(fps, 1)
    if actual <= target_duration_s + margin:
        return video_path
    trim_start = actual - target_duration_s
    trimmed_path = video_path.rsplit(".", 1)[0] + "_trimmed.mp4"
    cmd = ["ffmpeg", "-y", "-ss", f"{trim_start:.4f}", "-i", video_path,
           "-c:v", "libx264", "-c:a", "aac", "-pix_fmt", "yuv420p", trimmed_path]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0 and os.path.exists(trimmed_path):
        return trimmed_path
    return video_path

def split_prompts(text):
    if not text or not text.strip():
        return []
    normalized = text.replace("\r\n", "\n").replace("\r", "\n")
    blocks = re.split(r"\n[ \t]*\n+", normalized.strip())
    return [b.strip() for b in blocks if b.strip()]

def count_scenes(text):
    n = len(split_prompts(text))
    return f"🔹 **Số phân cảnh nhận diện được:** {n}"

def has_audio_stream(video_path):
    cmd = ["ffprobe", "-v", "error", "-select_streams", "a",
           "-show_entries", "stream=index", "-of", "csv=p=0", video_path]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        return bool(result.stdout.strip())
    except Exception:
        return True

def ensure_audio_track(video_path):
    if has_audio_stream(video_path):
        return video_path
    fixed_path = video_path.rsplit(".", 1)[0] + "_silentaudio.mp4"
    cmd = ["ffmpeg", "-y", "-i", video_path,
           "-f", "lavfi", "-i", "anullsrc=channel_layout=stereo:sample_rate=44100",
           "-shortest", "-c:v", "copy", "-c:a", "aac", "-map", "0:v:0", "-map", "1:a:0",
           fixed_path]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0 and os.path.exists(fixed_path):
        return fixed_path
    return video_path

def concat_videos(video_list, out_name, output_dir=OUTPUT_DIR):
    safe_video_list = [ensure_audio_track(v) for v in video_list]
    concat_file_path = os.path.join(output_dir, f"concat_{out_name}.txt")
    with open(concat_file_path, "w") as f:
        for vid in safe_video_list:
            f.write(f"file '{os.path.abspath(vid)}'\n")
    final_output = os.path.join(output_dir, f"{out_name}_{int(time.time())}.mp4")
    cmd = ["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", concat_file_path,
           "-c:v", "libx264", "-c:a", "aac", "-pix_fmt", "yuv420p", final_output]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0 or not os.path.exists(final_output):
        raise RuntimeError("Ghép nối video bằng ffmpeg thất bại.")
    return final_output

def _fmt_elapsed(s):
    m, sec = divmod(int(s), 60)
    return f"{m}m{sec:02d}s" if m else f"{sec}s"

_NODE_LABELS = {
    "UNETLoader": "📦 Loading Transformer",
    "CLIPLoader": "📦 Loading Text Encoder",
    "VAELoader": "📦 Loading VAE",
    "ComfyUILTX25MSRICLoRALoader": "🧬 Loading MSR LoRA",
    "PromptRelayEncode": "📝 Encoding Prompts",
    "CLIPTextEncode": "📝 Encoding Text",
    "LTXVConditioning": "⚙️ Conditioning",
    "ComfyUILTX25MSRMultiReferenceGuide": "🎭 MSR Reference Guide",
    "EmptyLTXVLatentVideo": "🎞️ Creating Latent",
    "LTXVEmptyLatentAudio": "🔊 Creating Audio Latent",
    "LTXVConcatAVLatent": "🔗 Concat AV Latent",
    "CFGGuider": "🎯 CFG Guider",
    "LTXVDualCFGGuider": "🎯 Dual CFG Guider",
    "RandomNoise": "🎲 Generating Noise",
    "SamplerCustomAdvanced": "🔥 SAMPLING (đang denoise)...",
    "LTXVSeparateAVLatent": "✂️ Separating AV",
    "LTXVCropGuides": "✂️ Cropping Guides",
    "VAEDecode": "🖼️ Decoding Video (Stage 1)",
    "VAEDecodeTiled": "🖼️ Decoding Video Tiled (Stage 2)",
    "LTXVAudioVAEDecode": "🔊 Decoding Audio",
    "CreateVideo": "🎬 Creating Video",
    "SaveVideo": "💾 Saving Video",
    "LTXVLatentUpsampler": "⬆️ Upscaling Latent x2",
    "LatentUpscaleModelLoader": "📦 Loading Upscaler",
    "ComfyMathExpression": "🔢 Computing Frames",
    "INTConstant": "⚙️ Setting Constants",
    "FloatConstant": "⚙️ Setting Constants",
    "KSamplerSelect": "⚙️ Selecting Sampler",
    "ManualSigmas": "⚙️ Setting Sigmas",
    "LoadImage": "🖼️ Loading Reference Image",
}

def submit_and_poll(workflow, scene_label="", max_wait_seconds=1800, poll_interval=3):
    """Generator: submit workflow, yield progress strings, return on completion."""
    data = json.dumps({"prompt": workflow}).encode("utf-8")
    req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)
    try:
        response = urllib.request.urlopen(req, timeout=30)
        prompt_id = json.loads(response.read())["prompt_id"]
    except urllib.error.HTTPError as e:
        try:
            body = e.read().decode("utf-8")
        except Exception:
            body = str(e)
        raise RuntimeError(f"ComfyUI từ chối workflow: {body[:800]}")
    except Exception as e:
        raise RuntimeError(f"Lỗi gửi job API: {e}")

    t0 = time.time()
    total_nodes = len(workflow)
    prev_msg = ""
    waited = 0

    while waited < max_wait_seconds:
        elapsed = time.time() - t0
        try:
            # Check if done
            history = json.loads(urllib.request.urlopen(
                urllib.request.Request(f"http://127.0.0.1:8188/history/{prompt_id}"), timeout=30).read())
            if str(prompt_id) in history:
                yield f"✅ Hoàn tất! ({_fmt_elapsed(elapsed)})"
                return prompt_id

            # Get queue info for progress
            queue = json.loads(urllib.request.urlopen(
                urllib.request.Request("http://127.0.0.1:8188/queue"), timeout=30).read())

            current_node_label = "⏳ Đang chờ..."
            nodes_done = 0
            is_running = False

            for job in queue.get("queue_running", []):
                if str(job[1]) == str(prompt_id):
                    is_running = True
                    if len(job) > 3 and isinstance(job[3], dict):
                        nodes_done = len(job[3].get("nodes_done", []))
                        cur_node_id = str(job[3].get("current_node", ""))
                        if cur_node_id and cur_node_id in workflow:
                            ct = workflow[cur_node_id].get("class_type", "")
                            current_node_label = _NODE_LABELS.get(ct, ct)
                    break

            if not is_running:
                for job in queue.get("queue_pending", []):
                    if str(job[1]) == str(prompt_id):
                        is_running = True
                        pending_pos = queue["queue_pending"].index(job)
                        current_node_label = f"📋 Đang chờ trong hàng đợi (vị trí {pending_pos + 1})"
                        break

            if not is_running:
                # Double-check history
                history2 = json.loads(urllib.request.urlopen(
                    urllib.request.Request(f"http://127.0.0.1:8188/history/{prompt_id}"), timeout=30).read())
                if str(prompt_id) in history2:
                    yield f"✅ Hoàn tất! ({_fmt_elapsed(elapsed)})"
                    return prompt_id
                raise RuntimeError(f"Render thất bại ở {scene_label}")

            pct = min(95, int(nodes_done / max(total_nodes, 1) * 100))
            filled = pct // 5
            bar = "█" * filled + "░" * (20 - filled)

            msg = (
                f"⏱️ {_fmt_elapsed(elapsed)} · {bar} {pct}%\n"
                f"📊 {nodes_done}/{total_nodes} nodes\n"
                f"{current_node_label}"
            )
            if msg != prev_msg:
                prev_msg = msg
                yield msg

        except RuntimeError:
            raise
        except Exception:
            raise RuntimeError(f"Server bị crash ở {scene_label}!")
        time.sleep(poll_interval)
        waited += poll_interval
    raise RuntimeError(f"Timeout: {scene_label} quá {max_wait_seconds // 60} phút.")

# ==========================================================================
# BUILD WORKFLOW MSR
# ==========================================================================
def build_msr_workflow(
    *,
    prompt_relay_desc,
    prompt_main,
    negative_text=None,
    width=1280, height=720,
    fps=24, duration=10,
    seed=None,
    video_cfg=1.5, audio_cfg=1.0,
    msr_lora_name=None,
    msr_lora_strength=0.85,
    pic1_name=None, pic2_name=None, pic3_name=None, pic4_name=None,
    background_name=None,
    msr_strength=0.7,
    reference_frames="33",
    use_tiled_encode=False,
    tile_size=256,
    run_stage2=True,
):
    if negative_text is None:
        negative_text = NEGATIVE_PROMPT_DEFAULT
    if msr_lora_name is None:
        msr_lora_name = MSR_LORA_FILENAME
    if seed is None:
        seed = random.randint(1, 999_999_999)

    safe_fps       = snap_fps_safe(fps)
    half_w, half_h = half_dims(width, height)

    pic_slot_map = [
        ("pic1",       pic1_name),
        ("pic2",       pic2_name),
        ("pic3",       pic3_name),
        ("pic4",       pic4_name),
        ("background", background_name),
    ]

    # ---- Stage 1 ----
    wf = {
        "S1_unet":  {"class_type": "UNETLoader",  "inputs": {"unet_name": UNET_FILENAME, "weight_dtype": "default"}},
        "S1_clip":  {"class_type": "CLIPLoader",  "inputs": {"clip_name": TEXT_ENCODER_FILENAME, "type": "ltxv", "device": "default"}},
        "S1_vvae":  {"class_type": "VAELoader",   "inputs": {"vae_name": VIDEO_VAE_FILENAME}},
        "S1_avae":  {"class_type": "VAELoader",   "inputs": {"vae_name": AUDIO_VAE_FILENAME}},
        "S1_msr_loader": {
            "class_type": "ComfyUILTX25MSRICLoRALoader",
            "inputs": {"model": ["S1_unet", 0], "lora_name": msr_lora_name, "strength_model": float(msr_lora_strength)},
        },
        "S1_neg_enc":     {"class_type": "CLIPTextEncode", "inputs": {"clip": ["S1_clip", 0], "text": negative_text}},
        "S1_width":       {"class_type": "INTConstant",    "inputs": {"value": half_w}},
        "S1_height":      {"class_type": "INTConstant",    "inputs": {"value": half_h}},
        "S1_fps":         {"class_type": "FloatConstant",  "inputs": {"value": float(safe_fps)}},
        "S1_frames_expr": {"class_type": "ComfyMathExpression", "inputs": {"expression": "a*b+1", "values.a": ["S1_fps", 0], "values.b": int(duration)}},
        "S1_empty_vid":   {"class_type": "EmptyLTXVLatentVideo",  "inputs": {"width": ["S1_width", 0], "height": ["S1_height", 0], "length": ["S1_frames_expr", 1], "batch_size": 1}},
        "S1_empty_aud":   {"class_type": "LTXVEmptyLatentAudio",  "inputs": {"audio_vae": ["S1_avae", 0], "frames_number": ["S1_frames_expr", 1], "frame_rate": ["S1_fps", 0], "batch_size": 1}},
        "S1_relay": {
            "class_type": "PromptRelayEncode",
            "inputs": {
                "model": ["S1_msr_loader", 0], "clip": ["S1_clip", 0], "latent": ["S1_empty_vid", 0],
                "global_prompt": prompt_relay_desc or "", "local_prompts": prompt_main,
                "segment_lengths": "", "epsilon": 0.001,
            },
        },
        "S1_ltxv_cond": {"class_type": "LTXVConditioning", "inputs": {"positive": ["S1_relay", 1], "negative": ["S1_neg_enc", 0], "frame_rate": ["S1_fps", 0]}},
    }

    safe_msr_strength = min(1.0, max(0.0, float(msr_strength)))
    msr_s1 = {
        "positive": ["S1_ltxv_cond", 0], "negative": ["S1_ltxv_cond", 1],
        "vae": ["S1_vvae", 0], "latent": ["S1_empty_vid", 0],
        "msr_parameters": ["S1_msr_loader", 1],
        "strength": safe_msr_strength, "reference_frames": reference_frames,
        "use_tiled_encode": use_tiled_encode, "tile_size": tile_size, "tile_overlap": 0,
    }
    for slot, img in pic_slot_map:
        if img:
            wf[f"S1_load_{slot}"] = {"class_type": "LoadImage", "inputs": {"image": img}}
            msr_s1[slot] = [f"S1_load_{slot}", 0]
    wf["S1_msr_guide"] = {"class_type": "ComfyUILTX25MSRMultiReferenceGuide", "inputs": msr_s1}

    wf.update({
        "S1_guider":      {"class_type": "CFGGuider",           "inputs": {"model": ["S1_relay", 0], "positive": ["S1_msr_guide", 0], "negative": ["S1_msr_guide", 1], "cfg": 1.0}},
        "S1_noise":       {"class_type": "RandomNoise",          "inputs": {"noise_seed": int(seed)}},
        "S1_sampler_sel": {"class_type": "KSamplerSelect",       "inputs": {"sampler_name": "euler_ancestral"}},
        "S1_sigmas":      {"class_type": "ManualSigmas",         "inputs": {"sigmas": SIGMAS_PASS1}},
        "S1_concat_av":   {"class_type": "LTXVConcatAVLatent",   "inputs": {"video_latent": ["S1_msr_guide", 2], "audio_latent": ["S1_empty_aud", 0]}},
        "S1_sample":      {"class_type": "SamplerCustomAdvanced", "inputs": {"noise": ["S1_noise", 0], "guider": ["S1_guider", 0], "sampler": ["S1_sampler_sel", 0], "sigmas": ["S1_sigmas", 0], "latent_image": ["S1_concat_av", 0]}},
        "S1_sep_av":      {"class_type": "LTXVSeparateAVLatent",  "inputs": {"av_latent": ["S1_sample", 0]}},
        "S1_crop_guides": {"class_type": "LTXVCropGuides",       "inputs": {"positive": ["S1_msr_guide", 0], "negative": ["S1_msr_guide", 1], "latent": ["S1_sep_av", 0]}},
        "S1_vae_decode":  {"class_type": "VAEDecode",             "inputs": {"samples": ["S1_crop_guides", 2], "vae": ["S1_vvae", 0]}},
        "S1_aud_decode":  {"class_type": "LTXVAudioVAEDecode",   "inputs": {"samples": ["S1_sep_av", 1], "audio_vae": ["S1_avae", 0]}},
        "S1_create_vid":  {"class_type": "CreateVideo",           "inputs": {"images": ["S1_vae_decode", 0], "audio": ["S1_aud_decode", 0], "fps": float(safe_fps)}},
        "S1_save":        {"class_type": "SaveVideo",             "inputs": {"video": ["S1_create_vid", 0], "filename_prefix": "output/LTX25_MSR_Stage1", "format": "auto", "codec": "auto"}},
    })

    if not run_stage2:
        return wf

    # ---- Stage 2 ----
    wf.update({
        "S2_upscale_loader": {"class_type": "LatentUpscaleModelLoader", "inputs": {"model_name": SPATIAL_UPSCALER_FILENAME}},
        "S2_upsampler":      {"class_type": "LTXVLatentUpsampler",       "inputs": {"samples": ["S1_crop_guides", 2], "upscale_model": ["S2_upscale_loader", 0], "vae": ["S1_vvae", 0]}},
        "S2_relay": {
            "class_type": "PromptRelayEncode",
            "inputs": {
                "model": ["S1_msr_loader", 0], "clip": ["S1_clip", 0], "latent": ["S2_upsampler", 0],
                "global_prompt": prompt_relay_desc or "", "local_prompts": prompt_main,
                "segment_lengths": "", "epsilon": 0.001,
            },
        },
    })

    stage2_seed = (int(seed) + 1000) if seed is not None else 42
    stage2_msr_strength = min(0.6, safe_msr_strength * 0.8)

    msr_s2 = {
        "positive": ["S2_relay", 1], "negative": ["S1_neg_enc", 0],
        "vae": ["S1_vvae", 0], "latent": ["S2_upsampler", 0],
        "msr_parameters": ["S1_msr_loader", 1],
        "strength": stage2_msr_strength, "reference_frames": reference_frames,
        "use_tiled_encode": use_tiled_encode, "tile_size": tile_size, "tile_overlap": 0,
    }
    for slot, img in pic_slot_map:
        if img:
            msr_s2[slot] = [f"S1_load_{slot}", 0]
    wf["S2_msr_guide"] = {"class_type": "ComfyUILTX25MSRMultiReferenceGuide", "inputs": msr_s2}

    wf.update({
        "S2_ltxv_cond":   {"class_type": "LTXVConditioning",      "inputs": {"positive": ["S2_msr_guide", 0], "negative": ["S2_msr_guide", 1], "frame_rate": ["S1_fps", 0]}},
        "S2_concat_av":   {"class_type": "LTXVConcatAVLatent",   "inputs": {"video_latent": ["S2_msr_guide", 2], "audio_latent": ["S1_sep_av", 1]}},
        "S2_dual_guider": {"class_type": "LTXVDualCFGGuider",    "inputs": {"model": ["S2_relay", 0], "positive": ["S2_ltxv_cond", 0], "negative": ["S2_ltxv_cond", 1], "video_cfg": float(video_cfg), "audio_cfg": float(audio_cfg)}},
        "S2_noise":       {"class_type": "RandomNoise",           "inputs": {"noise_seed": stage2_seed}},
        "S2_sampler_sel": {"class_type": "KSamplerSelect",        "inputs": {"sampler_name": "euler_ancestral"}},
        "S2_sigmas":      {"class_type": "ManualSigmas",          "inputs": {"sigmas": SIGMAS_PASS2}},
        "S2_sample":      {"class_type": "SamplerCustomAdvanced", "inputs": {"noise": ["S2_noise", 0], "guider": ["S2_dual_guider", 0], "sampler": ["S2_sampler_sel", 0], "sigmas": ["S2_sigmas", 0], "latent_image": ["S2_concat_av", 0]}},
        "S2_sep_av":      {"class_type": "LTXVSeparateAVLatent",  "inputs": {"av_latent": ["S2_sample", 0]}},
        "S2_crop_guides": {"class_type": "LTXVCropGuides",        "inputs": {"positive": ["S2_ltxv_cond", 0], "negative": ["S2_ltxv_cond", 1], "latent": ["S2_sep_av", 0]}},
        "S2_vae_tiled":   {"class_type": "VAEDecodeTiled",        "inputs": {"samples": ["S2_crop_guides", 2], "vae": ["S1_vvae", 0], "tile_size": 512, "overlap": 64, "temporal_size": 64, "temporal_overlap": 16}},
        "S2_aud_decode":  {"class_type": "LTXVAudioVAEDecode",   "inputs": {"samples": ["S2_sep_av", 1], "audio_vae": ["S1_avae", 0]}},
        "S2_create_vid":  {"class_type": "CreateVideo",           "inputs": {"images": ["S2_vae_tiled", 0], "audio": ["S2_aud_decode", 0], "fps": float(safe_fps)}},
        "S2_save":        {"class_type": "SaveVideo",             "inputs": {"video": ["S2_create_vid", 0], "filename_prefix": "output/LTX25_MSR_DualStage", "format": "auto", "codec": "auto"}},
    })

    return wf

# ==========================================================================
# GENERATE — GRADIO GENERATOR
# ==========================================================================
def generate_msr_gradio(
    pic1_path, pic2_path, pic3_path, pic4_path, background_path,
    prompt_relay_desc, prompt_main, negative_text,
    aspect_ratio, v_length, v_fps, v_seed, num_segments, fixed_seed,
    video_cfg, msr_lora_name, msr_lora_strength, msr_strength,
    reference_frames, use_tiled_encode,
    run_stage2, low_vram,
):
    if not pic1_path:
        yield None, None, "⚠️ Vui lòng tải ít nhất ảnh Pic 1!"; return

    prompts = split_prompts(prompt_main)
    if not prompts:
        yield None, None, "⚠️ Nhập ít nhất 1 prompt!"; return

    v_width, v_height = parse_aspect_ratio(aspect_ratio)
    safe_width, safe_height = safe_dims(v_width, v_height)

    yield None, None, "🔄 Đang khởi động ComfyUI server..."
    try:
        ensure_server(low_vram)
    except Exception as e:
        yield None, None, f"❌ {e}"; return

    base_seed = get_seed(v_seed)
    os.makedirs(INPUT_DIR, exist_ok=True)

    def _copy_img(path, slot_name):
        if not path:
            return None
        ext = os.path.splitext(path)[1].lower() or ".png"
        name = f"msr_{slot_name}_{int(time.time())}{ext}"
        shutil.copy(path, os.path.join(INPUT_DIR, name))
        return name

    pic1_name = _copy_img(pic1_path, "pic1")
    pic2_name = _copy_img(pic2_path, "pic2")
    pic3_name = _copy_img(pic3_path, "pic3")
    pic4_name = _copy_img(pic4_path, "pic4")
    bg_name   = _copy_img(background_path, "background")

    if len(prompts) > 1:
        scene_prompts = prompts
    else:
        num_segments = max(1, int(num_segments))
        scene_prompts = [prompts[0]] * num_segments

    total_scenes = len(scene_prompts)
    total_seconds = total_scenes * int(v_length)
    stage_note = "Stage 1 + Stage 2 (upscale x2)" if run_stage2 else "Stage 1 only"

    yield None, None, (
        f"✅ Server sẵn sàng. Bắt đầu {total_scenes} phân cảnh ({total_seconds}s)...\n"
        f"Chế độ: {stage_note} · Base Seed: {base_seed} · Video CFG: {video_cfg}"
    )

    generated_videos = []
    for i, p in enumerate(scene_prompts):
        label = f"phân cảnh {i + 1}/{total_scenes}"
        seed_i = base_seed if fixed_seed else (base_seed + i)

        yield generated_videos, None, (
            f"🔄 Đang quay {label} [{stage_note}]... (Seed: {seed_i})\n"
            f"📝 {p[:120]}..."
        )

        wf = build_msr_workflow(
            prompt_relay_desc = prompt_relay_desc or "",
            prompt_main       = p,
            negative_text     = negative_text or NEGATIVE_PROMPT_DEFAULT,
            width             = safe_width,
            height            = safe_height,
            fps               = v_fps,
            duration          = v_length,
            seed              = seed_i,
            video_cfg         = float(video_cfg or 1.5),
            msr_lora_name     = msr_lora_name,
            msr_lora_strength = msr_lora_strength,
            pic1_name         = pic1_name,
            pic2_name         = pic2_name,
            pic3_name         = pic3_name,
            pic4_name         = pic4_name,
            background_name   = bg_name,
            msr_strength      = msr_strength,
            reference_frames  = str(reference_frames),
            use_tiled_encode  = bool(use_tiled_encode),
            run_stage2        = bool(run_stage2),
        )

        try:
            for progress_msg in submit_and_poll(wf, scene_label=label):
                yield generated_videos, None, (
                    f"🎬 {label} · Seed: {seed_i}\n"
                    f"{progress_msg}"
                )
        except Exception as e:
            yield generated_videos, None, f"❌ {e}"; return

        latest_video = find_latest_video()
        if not latest_video:
            yield generated_videos, None, f"⚠️ Không tìm thấy video ở {label}!"; return

        latest_video = trim_ref_frames(latest_video, target_duration_s=int(v_length), fps=v_fps)
        generated_videos.append(latest_video)
        yield generated_videos, None, f"🔔 [DING] ✅ Xong {label}!"

    if len(generated_videos) > 1:
        yield generated_videos, None, "🔄 Đang ghép nối các phân cảnh..."
        try:
            final_output = concat_videos(generated_videos, "final_long_msr_video")
        except Exception as e:
            yield generated_videos, generated_videos[-1], f"⚠️ {e}"; return
        yield generated_videos, final_output, (
            f"🔔 [DING] 🎉 Hoàn tất phim MSR ({total_seconds}s, {total_scenes} cảnh)! Seed: {base_seed}"
        )
    elif len(generated_videos) == 1:
        yield generated_videos, generated_videos[0], (
            f"🔔 [DING] 🎉 Render Complete! ({v_length}s) Seed: {base_seed}"
        )

# ==========================================================================
# GRADIO UI
# ==========================================================================
ratio_choices = [
    "16:9 (1280x720) · HD Ngang",
    "9:16 (720x1280) · HD Dọc",
    "1:1 (720x720) · Vuông",
    "16:9 (832x480) · Nhẹ VRAM",
    "9:16 (480x832) · Nhẹ VRAM",
]

notification_js = """
function(){
    let last="";
    function ding(){
        try{
            let c=new(window.AudioContext||window.webkitAudioContext)();
            let o=c.createOscillator(),g=c.createGain();
            o.type='sine'; o.frequency.setValueAtTime(880,c.currentTime);
            o.frequency.exponentialRampToValueAtTime(1760,c.currentTime+.15);
            g.gain.setValueAtTime(.3,c.currentTime);
            g.gain.exponentialRampToValueAtTime(.01,c.currentTime+.4);
            o.connect(g);g.connect(c.destination);o.start();o.stop(c.currentTime+.4);
        }catch(e){}
    }
    new MutationObserver(()=>{
        document.querySelectorAll('.status-box textarea').forEach(tb=>{
            let t=tb.value||"";
            if(t.includes('[DING]')&&t!==last){last=t;ding();}
        });
    }).observe(document.body,{childList:true,subtree:true,characterData:true});
}
"""

custom_css = """
.gradio-container { max-width: 1400px; margin: 0 auto; }
#msr-header { background: linear-gradient(135deg, #7c3aed 0%, #a855f7 100%);
              border-radius:16px; padding:20px 26px; margin-bottom:14px; }
#msr-header h1, #msr-header p, #msr-header h3, #msr-header div { color:#fff !important; margin:0 !important; }
.info-box { background:rgba(99,102,241,.08); border-left:3px solid #6366f1;
            padding:10px 14px; border-radius:8px; font-size:.87rem; margin-bottom:8px; }
.scene-counter { display:inline-block; background:rgba(99,102,241,.12); padding:6px 14px; border-radius:999px; font-weight:600 !important; font-size:.85rem !important; }
.scene-counter p { margin:0 !important; color:#4f46e5 !important; }
.status-box textarea { font-family:monospace !important; font-size:.82rem !important; }
"""

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="violet", secondary_hue="purple", neutral_hue="slate"),
    title="LTX-2.5 MSR Studio",
    css=custom_css,
    js=notification_js,
    fill_width=True,
) as demo:

    with gr.Column(elem_id="msr-header"):
        gr.Markdown(
            "### 🎭 LTX 2.5-MSR — Multi-Subject Reference Video Studio\n"
            "Tạo phim dài nhiều phân cảnh từ ảnh tham khảo nhân vật & bối cảnh\n\n"
            "🎭 Tối đa 4 nhân vật + 1 bối cảnh · 🎞️ Auto render & ghép nối bằng ffmpeg"
        )

    with gr.Row():
        with gr.Column(scale=5):
            with gr.Group():
                gr.Markdown("### 📸 Ảnh tham khảo")
                gr.Markdown(
                    "<div class='info-box'>Chỉ <b>Pic 1</b> bắt buộc, các slot còn lại tuỳ chọn.</div>"
                )
                with gr.Row():
                    msr_pic1 = gr.Image(label="🎭 Pic 1 (bắt buộc)", type="filepath")
                    msr_pic2 = gr.Image(label="🎭 Pic 2 (tuỳ chọn)", type="filepath")
                with gr.Row():
                    msr_pic3 = gr.Image(label="🎭 Pic 3 (tuỳ chọn)", type="filepath")
                    msr_pic4 = gr.Image(label="🎭 Pic 4 (tuỳ chọn)", type="filepath")
                msr_bg = gr.Image(label="🌄 Background (tuỳ chọn)", type="filepath")

            with gr.Group():
                gr.Markdown("### 📝 Kịch bản & Prompt")
                gr.Markdown(
                    "<div class='info-box'>"
                    "① <b>Mô tả nhân vật</b>: Image 1:... Image 2:...<br>"
                    "② <b>Kịch bản</b>: nhiều cảnh cách nhau 1 dòng trống. Auto ghép nối!"
                    "</div>"
                )
                msr_relay_desc = gr.Textbox(
                    label="① Mô tả nhân vật", lines=4,
                    placeholder="Image 1: A real chubby orange tabby cat...\n\nImage 2: A cute Corgi puppy...")
                scene_count_display = gr.Markdown("🔹 **Số phân cảnh:** 0", elem_classes="scene-counter")
                msr_prompt = gr.Textbox(
                    label="② Kịch bản (mỗi cảnh cách 1 dòng trống)", lines=6,
                    placeholder="[Shot 1] The cat gestures with a spoon...\n\n[Shot 2] The corgi slips on the floor...")
                msr_neg = gr.Textbox(label="🚫 Negative", lines=2, value=NEGATIVE_PROMPT_DEFAULT)

            with gr.Accordion("⚙️ Cài đặt nâng cao", open=False):
                gr.Markdown("**📐 Kích thước & thời lượng**")
                ratio_msr = gr.Radio(label="Tỉ lệ", choices=ratio_choices, value=ratio_choices[0])
                with gr.Row():
                    length_msr = gr.Slider(label="⏱️ Giây/cảnh", minimum=1, maximum=20, step=1, value=10)
                    fps_msr = gr.Slider(label="🎞️ FPS", minimum=8, maximum=120, step=8, value=24)
                with gr.Row():
                    seed_msr = gr.Number(label="🎲 Seed (-1=random)", value=-1, precision=0)
                    num_segments_msr = gr.Slider(label="🔢 Số segment (1 prompt)", minimum=1, maximum=10, step=1, value=1)
                fixed_seed_msr = gr.Checkbox(label="🔗 Cùng Seed mọi cảnh", value=False)
                gr.Markdown("**🧬 MSR LoRA**")
                with gr.Row():
                    msr_lora_dd = gr.Dropdown(label="MSR LoRA", choices=list_msr_loras(), value=MSR_LORA_FILENAME, scale=3)
                    msr_lora_str_sl = gr.Slider(label="LoRA strength", minimum=0.0, maximum=2.0, step=0.05, value=0.85, scale=2)
                msr_refresh_btn = gr.Button("🔄 Refresh LoRA", size="sm")
                msr_refresh_btn.click(fn=lambda: gr.update(choices=list_msr_loras()), outputs=[msr_lora_dd])
                gr.Markdown("**🎯 MSR Guide & CFG**")
                with gr.Row():
                    msr_video_cfg = gr.Slider(label="Video CFG", minimum=1.0, maximum=3.5, step=0.1, value=1.5, scale=1)
                    msr_guide_str_sl = gr.Slider(label="Ref strength", minimum=0.0, maximum=1.0, step=0.05, value=0.7, scale=1)
                with gr.Row():
                    msr_ref_frames = gr.Radio(label="Ref frames", choices=["25", "33"], value="33")
                    msr_tiled = gr.Checkbox(label="Tiled VAE encode", value=False)
                gr.Markdown("**⚙️ Pipeline**")
                with gr.Row():
                    msr_stage2 = gr.Checkbox(label="✅ Stage 2 (upscale x2)", value=True)
                    msr_low_vram = gr.Checkbox(label="🧊 Low VRAM", value=True)

        with gr.Column(scale=5):
            with gr.Group():
                gallery_msr = gr.Gallery(label="🎥 Các Phân Cảnh Lẻ", columns=2, height="auto")
                video_out_msr = gr.Video(label="🎬 Phim Hoàn Chỉnh")
                with gr.Row():
                    msr_btn = gr.Button("🎬 Bắt Đầu Tạo Phim MSR", variant="primary", scale=3)
                    msr_clear = gr.Button("🗑️ Clear", scale=1)
                msr_status = gr.Textbox(
                    label="ℹ️ Status", interactive=False, lines=5, elem_classes="status-box")
            gr.Markdown(
                "<div class='info-box'>"
                "<b>💡 Tạo phim 30s–60s:</b> Dán nhiều cảnh (cách 2 Enter). "
                "Hệ thống tự quay từng cảnh rồi ghép nối thành video hoàn chỉnh!"
                "</div>"
            )

    msr_prompt.change(fn=count_scenes, inputs=[msr_prompt], outputs=[scene_count_display])
    msr_btn.click(
        fn=generate_msr_gradio,
        inputs=[
            msr_pic1, msr_pic2, msr_pic3, msr_pic4, msr_bg,
            msr_relay_desc, msr_prompt, msr_neg,
            ratio_msr, length_msr, fps_msr, seed_msr, num_segments_msr, fixed_seed_msr,
            msr_video_cfg, msr_lora_dd, msr_lora_str_sl, msr_guide_str_sl,
            msr_ref_frames, msr_tiled,
            msr_stage2, msr_low_vram,
        ],
        outputs=[gallery_msr, video_out_msr, msr_status],
    )
    msr_clear.click(
        fn=lambda: (None, None, "", "🔹 **Số phân cảnh:** 0"),
        outputs=[gallery_msr, video_out_msr, msr_status, scene_count_display],
    )

demo.queue()
demo.launch(share=True, inline=False, debug=True)